# KrishiSetu AI - Edge Model Training for PWA
This notebook trains a lightweight MobileNetV2 model to detect crop diseases and exports it to TensorFlow.js format so it can run 100% offline in our React webapp.

### Instructions:
1. Upload your dataset (folders of images organized by disease name) from data.gov.in or AI Kosh.
2. Set the `dataset_dir` path below.
3. Run all cells.
4. Download the `tfjs_model` folder generated at the end and place it in the `public/model/` folder of your React app.

In [ ]:
!pip install tensorflowjs
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import tensorflowjs as tfjs
import os

In [ ]:
# Define Dataset Path and Params
dataset_dir = '/content/dataset' # CHANGE THIS to your extracted dataset folder
img_size = (224, 224)
batch_size = 32

print("Loading dataset...")
train_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=img_size,
  batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=img_size,
  batch_size=batch_size)

class_names = train_ds.class_names
print("Classes detected:", class_names)

In [ ]:
# Build the lightweight model
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base model for transfer learning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model (Keep epochs low for hackathon speed)
model.fit(train_ds, validation_data=val_ds, epochs=5)

In [ ]:
# Export for the Web App!
tfjs_target_dir = '/content/tfjs_model'
tfjs.converters.save_keras_model(model, tfjs_target_dir)

# Save the class names so the app knows what it predicted
with open(os.path.join(tfjs_target_dir, 'classes.json'), 'w') as f:
    import json
    json.dump(class_names, f)

print(f"SUCCESS! Download the {tfjs_target_dir} folder and put it in KrishiSetu-AI/public/model/")